# 📈 03. Exploratory Data Analysis (EDA)

This notebook runs univariate, bivariate, multivariate, correlation, and outlier analyses on credit card application datasets, saving visualization plots directly to the screenshots folder.

## 1. Project Introduction & Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append project root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.data.load_data import DataLoader
from src.features.feature_engineering import FeatureEngineer
from src.visualization.plots import VizPlotter
from configs.constants import TARGET_COL

%matplotlib inline
print("Libraries loaded successfully.")

## 2. Load Dataset

In [ ]:
loader = DataLoader()
app_df, credit_df = loader.load_all()

# Derive target variable
bad_statuses = {'2', '3', '4', '5'}
credit_df['IS_BAD'] = credit_df['STATUS'].astype(str).apply(lambda x: 1 if x in bad_statuses else 0)
target_df = credit_df.groupby('ID')['IS_BAD'].max().reset_index()
target_df.rename(columns={'IS_BAD': TARGET_COL}, inplace=True)

# Custom features
engineer = FeatureEngineer()
app_cleaned = engineer.extract_custom_features(app_df)
merged_df = pd.merge(app_cleaned, target_df, on='ID', how='inner')

print(f"Merged Dataset Shape: {merged_df.shape}")

## 3. Basic Dataset Information
Displaying head, tail, sample records, dtypes, missing fields, and duplication checks.

In [ ]:
print("--- Dataset Head ---")
print(merged_df.head(2))

print("\n--- Dataset Tail ---")
print(merged_df.tail(2))

print("\n--- Random Sample ---")
print(merged_df.sample(2))

print("\n--- Missing Values ---")
print(merged_df.isnull().sum())

print(f"\nDuplicate rows: {merged_df.duplicated().sum()}")

## 4. Statistical Summary

In [ ]:
print("--- Numerical Features summary ---")
print(merged_df.describe().transpose())

print("\n--- Categorical Features summary ---")
print(merged_df.describe(include=['object']).transpose())

## 5. Univariate Analysis (Visualizations)

In [ ]:
plotter = VizPlotter()

# Target class distribution
plotter.plot_target_balance(merged_df[TARGET_COL], "approval_count.png")

# Income distribution
plotter.plot_distribution(merged_df, "AMT_INCOME_TOTAL", "income_distribution.png")

# Employment years distribution
plotter.plot_distribution(merged_df, "YEARS_EMPLOYED", "employment_distribution.png")

## 6. Bivariate Analysis

In [ ]:
# Education vs target
plotter.plot_categorical_vs_target(merged_df, "NAME_EDUCATION_TYPE", TARGET_COL, "education_vs_approval.png")

# Gender vs target
plotter.plot_categorical_vs_target(merged_df, "CODE_GENDER", TARGET_COL, "gender_vs_approval.png")

# Housing vs target
plotter.plot_categorical_vs_target(merged_df, "NAME_HOUSING_TYPE", TARGET_COL, "housing_vs_approval.png")

## 7. Correlation Analysis

In [ ]:
# Heatmap
plotter.plot_correlation_heatmap(merged_df, "correlation_heatmap.png")

## 8. Outlier Detection

In [ ]:
# Outlier Boxplots
plotter.plot_outliers_boxplot(merged_df, "AMT_INCOME_TOTAL", "outlier_boxplot_income.png")
plotter.plot_outliers_boxplot(merged_df, "AGE_YEARS", "outlier_boxplot_age.png")

## 9. Business Insights & Conclusions

- **Default Target**: Imbalanced profile. (Class 0: 92.5%, Class 1: 7.5%).
- **Income & Employment**: Crucial indicators for risk assessment. Retired pensioners segment holds high sample weight.
- **Collateral proxy**: Real estate and car ownership correlate with reduced default risk.